# Step 2: Data Cleaning

Cleaning steps based on EDA findings:
1. Load all raw files
2. Clean `oil.csv` — forward-fill + backward-fill missing prices
3. Clean `holidays_events.csv` — handle `transferred=True` rows
4. Clean `transactions.csv` — fill 2 missing dates
5. Merge all tables into a single train/test dataframe
6. Final validation
7. Save cleaned data to Parquet

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

DATA_DIR   = 'data/'
OUTPUT_DIR = 'data/'

## 1. Load Raw Data

In [2]:
train        = pd.read_csv(DATA_DIR + 'train.csv',           parse_dates=['date'])
test         = pd.read_csv(DATA_DIR + 'test.csv',            parse_dates=['date'])
stores       = pd.read_csv(DATA_DIR + 'stores.csv')
oil          = pd.read_csv(DATA_DIR + 'oil.csv',             parse_dates=['date'])
holidays     = pd.read_csv(DATA_DIR + 'holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv(DATA_DIR + 'transactions.csv',    parse_dates=['date'])

print('Loaded:')
for name, df in [('train', train), ('test', test), ('stores', stores),
                 ('oil', oil), ('holidays', holidays), ('transactions', transactions)]:
    print(f'  {name:12s}: {df.shape}')

Loaded:
  train       : (3000888, 6)
  test        : (28512, 5)
  stores      : (54, 5)
  oil         : (1218, 2)
  holidays    : (350, 6)
  transactions: (83488, 3)


## 2. Clean Oil Prices

43 missing days (weekends/holidays when markets are closed).  
Reindex to a daily calendar, then `ffill` + `bfill` (bfill handles the single leading NaN on 2013-01-01).

In [3]:
oil_clean = (
    oil
    .set_index('date')
    .resample('D').first()       # expand to daily calendar
    .ffill()                     # carry last known price forward
    .bfill()                     # fill the single leading NaN (2013-01-01)
    .reset_index()
    .rename(columns={'dcoilwtico': 'oil_price'})
)

assert oil_clean['oil_price'].isnull().sum() == 0, "Oil still has NaN!"
print(f'oil_clean: {oil_clean.shape}, NaN = {oil_clean["oil_price"].isnull().sum()}')
oil_clean.head()

oil_clean: (1704, 2), NaN = 0


,date,oil_price
0,2013-01-01,93.14
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-05,93.12


## 3. Clean Holidays

Two edge cases:
- **`transferred=True`**: the holiday on that date was *moved* to another day — treat as a regular working day.
- **`type='Transfer'`**: the actual replacement holiday date — treat as a holiday.

We build three holiday flags used later in feature engineering:
- `is_national_holiday`: national-level holiday on that date
- `is_regional_holiday`: regional-level
- `is_local_holiday`: local-level (city-specific)

We also keep `holiday_type` for the national holidays.

In [4]:
# Keep only effective holidays (drop rows where transferred=True, i.e. the moved-away date)
holidays_clean = holidays[holidays['transferred'] == False].copy()

# Rename 'Transfer' type to 'Holiday' so downstream code treats it uniformly
holidays_clean['type'] = holidays_clean['type'].replace('Transfer', 'Holiday')

print('Holiday types after cleaning:')
print(holidays_clean['type'].value_counts())
print('\nLocale distribution:')
print(holidays_clean['locale'].value_counts())

Holiday types after cleaning:
type
Holiday       221
Event          56
Additional     51
Bridge          5
Work Day        5
Name: count, dtype: int64

Locale distribution:
locale
National    166
Local       148
Regional     24
Name: count, dtype: int64


In [5]:
# Build date-level holiday flags for each locale
def make_holiday_flag(df, locale):
    sub = df[df['locale'] == locale][['date', 'type', 'locale_name']].drop_duplicates('date')
    sub = sub.rename(columns={
        'type':        f'{locale.lower()}_holiday_type',
        'locale_name': f'{locale.lower()}_holiday_name'
    })
    sub[f'is_{locale.lower()}_holiday'] = True
    return sub

nat_hol  = make_holiday_flag(holidays_clean, 'National')
reg_hol  = make_holiday_flag(holidays_clean, 'Regional')
loc_hol  = make_holiday_flag(holidays_clean, 'Local')

print(f'National holiday dates: {len(nat_hol)}')
print(f'Regional holiday dates: {len(reg_hol)}')
print(f'Local holiday dates:    {len(loc_hol)}')

National holiday dates: 160
Regional holiday dates: 24
Local holiday dates:    134


## 4. Clean Transactions

`transactions` is missing 2 dates (2016-01-01, 2016-01-03).  
We fill those with 0 — stores were likely closed and had no transactions.

In [6]:
# Build a full date × store grid so no (date, store) pair is missing
all_dates  = pd.date_range(transactions['date'].min(), transactions['date'].max(), freq='D')
all_stores = transactions['store_nbr'].unique()
full_grid  = pd.MultiIndex.from_product([all_dates, all_stores], names=['date', 'store_nbr'])

transactions_clean = (
    transactions
    .set_index(['date', 'store_nbr'])
    .reindex(full_grid, fill_value=0)
    .reset_index()
)

missing_before = len(all_dates) * len(all_stores) - len(transactions)
print(f'Rows added to fill gaps: {len(transactions_clean) - len(transactions)}')
print(f'transactions_clean shape: {transactions_clean.shape}')
assert transactions_clean['transactions'].isnull().sum() == 0

Rows added to fill gaps: 7664
transactions_clean shape: (91152, 3)


## 5. Merge All Tables

Merge order:
```
train/test
  + stores          (on store_nbr)       → city, state, store_type, cluster
  + oil_clean       (on date)            → oil_price
  + nat_hol         (on date)            → is_national_holiday, national_holiday_type
  + reg_hol         (on date)            → is_regional_holiday, regional_holiday_name
  + loc_hol         (on date)            → is_local_holiday, local_holiday_name
  + transactions    (on date+store_nbr)  → transactions
```

In [7]:
def merge_all(df):
    df = df.copy()

    # stores
    df = df.merge(stores.rename(columns={'type': 'store_type'}), on='store_nbr', how='left')

    # oil
    df = df.merge(oil_clean[['date', 'oil_price']], on='date', how='left')

    # national holidays
    df = df.merge(nat_hol, on='date', how='left')
    df['is_national_holiday'] = df['is_national_holiday'].fillna(False)

    # regional holidays — match by state
    df = df.merge(
        reg_hol.rename(columns={'regional_holiday_name': 'reg_holiday_name'}),
        on='date', how='left'
    )
    df['is_regional_holiday'] = df['is_regional_holiday'].fillna(False)

    # local holidays — match by city (locale_name == city)
    df = df.merge(
        loc_hol.rename(columns={'local_holiday_name': 'loc_holiday_name'}),
        on='date', how='left'
    )
    df['is_local_holiday'] = df['is_local_holiday'].fillna(False)

    # transactions
    df = df.merge(transactions_clean, on=['date', 'store_nbr'], how='left')
    df['transactions'] = df['transactions'].fillna(0).astype(int)

    return df

train_merged = merge_all(train)
test_merged  = merge_all(test)

print(f'train_merged: {train_merged.shape}')
print(f'test_merged:  {test_merged.shape}')

train_merged: (3000888, 21)
test_merged:  (28512, 20)


## 6. Final Validation

In [8]:
# Check dtypes
print('=== train_merged dtypes ===')
print(train_merged.dtypes)

=== train_merged dtypes ===
id                                int64
date                     datetime64[ns]
store_nbr                         int64
family                           object
sales                           float64
onpromotion                       int64
city                             object
state                            object
store_type                       object
cluster                           int64
oil_price                       float64
national_holiday_type            object
national_holiday_name            object
is_national_holiday                bool
regional_holiday_type            object
reg_holiday_name                 object
is_regional_holiday                bool
local_holiday_type               object
loc_holiday_name                 object
is_local_holiday                   bool
transactions                      int64
dtype: object


In [9]:
# Missing value report
def missing_report(df, name):
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print(f'{name}: no missing values ✓')
    else:
        print(f'{name} — columns with missing values:')
        pct = (missing / len(df) * 100).round(2)
        print(pd.DataFrame({'missing': missing, 'pct': pct}))

missing_report(train_merged, 'train_merged')
missing_report(test_merged,  'test_merged')

train_merged — columns with missing values:
                       missing    pct
national_holiday_type  2758536  91.92
national_holiday_name  2758536  91.92
regional_holiday_type  2968812  98.93
reg_holiday_name       2968812  98.93
local_holiday_type     2815560  93.82
loc_holiday_name       2815560  93.82
test_merged — columns with missing values:
                       missing     pct
national_holiday_type    28512  100.00
national_holiday_name    28512  100.00
regional_holiday_type    28512  100.00
reg_holiday_name         28512  100.00
local_holiday_type       26730   93.75
loc_holiday_name         26730   93.75


In [10]:
# Core sanity checks
assert (train_merged['sales'] < 0).sum() == 0,        'Negative sales found!'
assert train_merged['oil_price'].isnull().sum() == 0,  'Oil price has NaN!'
assert train_merged['store_type'].isnull().sum() == 0, 'store_type has NaN!'
assert train_merged['cluster'].isnull().sum() == 0,    'cluster has NaN!'

print('All sanity checks passed ✓')
print(f'\nDate range — train: {train_merged["date"].min().date()} -> {train_merged["date"].max().date()}')
print(f'Date range — test:  {test_merged["date"].min().date()} -> {test_merged["date"].max().date()}')
print(f'\nHoliday flags (train):')
print(train_merged[['is_national_holiday','is_regional_holiday','is_local_holiday']].sum())

All sanity checks passed ✓

Date range — train: 2013-01-01 -> 2017-08-15
Date range — test:  2017-08-16 -> 2017-08-31

Holiday flags (train):
is_national_holiday    242352
is_regional_holiday     32076
is_local_holiday       185328
dtype: int64


In [11]:
# Preview final schema
train_merged.head(3)

,id,date,store_nbr,family,sales,onpromotion,city,state,store_type,cluster,...,national_holiday_type,national_holiday_name,is_national_holiday,regional_holiday_type,reg_holiday_name,is_regional_holiday,local_holiday_type,loc_holiday_name,is_local_holiday,transactions
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,...,Holiday,Ecuador,True,NaN,NaN,False,NaN,NaN,False,0
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,...,Holiday,Ecuador,True,NaN,NaN,False,NaN,NaN,False,0
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,...,Holiday,Ecuador,True,NaN,NaN,False,NaN,NaN,False,0


## 7. Save Cleaned Data

In [12]:
train_merged.to_parquet(OUTPUT_DIR + 'train_cleaned.parquet', index=False)
test_merged.to_parquet(OUTPUT_DIR  + 'test_cleaned.parquet',  index=False)

print('Saved:')
print(f'  data/train_cleaned.parquet  ({train_merged.shape[0]:,} rows x {train_merged.shape[1]} cols)')
print(f'  data/test_cleaned.parquet   ({test_merged.shape[0]:,} rows x {test_merged.shape[1]} cols)')
print('\nColumns in cleaned data:')
print(list(train_merged.columns))

Saved:
  data/train_cleaned.parquet  (3,000,888 rows x 21 cols)
  data/test_cleaned.parquet   (28,512 rows x 20 cols)

Columns in cleaned data:
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'store_type', 'cluster', 'oil_price', 'national_holiday_type', 'national_holiday_name', 'is_national_holiday', 'regional_holiday_type', 'reg_holiday_name', 'is_regional_holiday', 'local_holiday_type', 'loc_holiday_name', 'is_local_holiday', 'transactions']
